# Ajuste de hiperparámetros en fine tunning

Instalar paquetes necesarios

In [ ]:
!pip install biopython
!pip install keras-pos-embd
!pip install keras_layer_normalization
!pip install keras_transformer
!pip install keras-bert
!pip install keras_tuner

Importar las librerías necesarias

In [ ]:
import tensorflow as tf
from tensorflow import keras
import json
from preprocessing.process_inputs import get_class_vectors, ALPHABET
from model import PARAMS
import numpy as np
from tensorflow.keras.utils import Sequence
from bert_utils import get_token_dict, seq2tokens, predict
from bert_utils import generate_bert_with_pretrained, generate_bert_with_pretrained_multi_tax, \
    get_classes_and_weights_multi_tax
from random import shuffle, sample
from sklearn.model_selection import train_test_split
import os
import argparse
from dataclasses import dataclass, field
from typing import List, Optional
from logging import warning
import pickle
from tensorflow.keras import mixed_precision
from tensorflow.keras.callbacks import ModelCheckpoint, TensorBoard, EarlyStopping
from tensorflow.keras.callbacks import Callback
from os.path import splitext
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from keras.models import load_model
import keras_tuner as kt
from dependencies.keras_bert.keras_bert.loader import build_model_from_config
import time

Vincular con Google Driver

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Comprobar uso  de GPUs

In [ ]:
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

Num GPUs Available:  1


Declaración de variable global classes para clasificación binaria de Cacao y No cacao

In [ ]:
classes = ["NotCacao", "Cacao"]

Declaración de clases y funciones:

**Clases**:

*   LossHistoryLogger
*   FragmentGenerator

**Funciones**:

*   load_dataset
*   load_fragments
*   get_fine_model
*   build_model

In [ ]:
class LossHistoryLogger(Callback):
    def __init__(self, filename='loss_history_fine_tune.csv'):
        super().__init__()
        self.filename = filename
        self.history = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        logs['epoch'] = epoch + 1
        self.history.append(logs)
        df = pd.DataFrame(self.history)
        df.to_csv(self.filename, index=False)

In [ ]:
def load_dataset(filepath):
    df = pd.read_csv(filepath, sep="\t")
    x = df["x"]
    y = df["y"]
    y_species = df["tax_id"]
    return x, y, y_species

In [ ]:
def load_fragments(fragments_dir, shuffle_=True, balance=True, nr_seqs=None):
    fragments = []
    species_list = []
    for class_ in classes:
        fragments.append((class_, json.load(open(os.path.join(
            fragments_dir, f'{class_}_fragments.json')))))
        species_list.append([int(line.strip()) for line in
                             open(os.path.join(fragments_dir, f'{class_}_species_picked.txt')).readlines()])
    nr_seqs_max = min(len(item[1]) for item in fragments)
    if (nr_seqs is None or nr_seqs > nr_seqs_max):
        nr_seqs = nr_seqs_max
    x = []
    y = np.array([])
    y_species = np.array([], dtype=int)

    for index, fragments_i in enumerate(fragments):
        class_, class_fragments = fragments_i
        if not balance:
            x.extend(class_fragments)
            y = np.append(y, [class_] * len(class_fragments))
            y_species = np.append(y_species, species_list[index])
        else:
            x_help = list(zip(class_fragments, species_list[index]))
            # x.extend(sample(class_fragments, nr_seqs))
            x_help = sample(x_help, nr_seqs)
            x_help, y_species_help = zip(*x_help)
            x.extend(x_help)
            y_species = np.append(y_species, y_species_help)
            y = np.append(y, [class_] * nr_seqs)

    assert len(x) == len(y)
    if (shuffle_):
        to_shuffle = list(zip(x, y, y_species))
        shuffle(to_shuffle)
        x, y, y_species = zip(*to_shuffle)
    # SOBREESCRIBIR LAS ETIQUETAS BASÁNDONOS EN tax_id:
    y = np.array(["Cacao" if taxid == 3641 else "NotCacao" for taxid in y_species])

    print(f'{len(x)} fragments loaded in total; '
          f'balanced={balance}, shuffle_={shuffle_}, nr_seqs={nr_seqs}')
    return np.array(x), np.array(y), np.array(y_species)

In [ ]:
@dataclass
class FragmentGenerator(Sequence):
    x: list
    y: list
    seq_len: int
    max_seq_len: Optional[int] = None
    k: int = 3
    stride: int = 3
    batch_size: int = 32
    classes: List = field(default_factory=lambda:
    ["NotCacao", "Cacao"])
    seq_len_like: Optional[np.array] = None
    window: bool = False

    def __post_init__(self):
        self.class_vectors = get_class_vectors(self.classes)
        self.token_dict = get_token_dict(ALPHABET, k=3)
        if (self.max_seq_len is None):
            self.max_seq_len = self.seq_len

    def __len__(self):
        return np.ceil(len(self.x)
                       / float(self.batch_size)).astype(int)

    def __getitem__(self, idx):
      batch_fragments = self.x[idx * self.batch_size:
                              (idx + 1) * self.batch_size]
      batch_x = [seq2tokens(seq, self.token_dict, seq_length=self.seq_len,
                            max_length=self.max_seq_len,
                            k=self.k, stride=self.stride, window=self.window,
                            seq_len_like=self.seq_len_like)
                for seq in batch_fragments]

      x_inputs = (
          np.array([item[0] for item in batch_x]),  # Token input
          np.array([item[1] for item in batch_x])   # Segment input
      )

      if self.y is not None and len(self.y) != 0:
          batch_classes = self.y[idx * self.batch_size:
                                (idx + 1) * self.batch_size]
          batch_y = np.array([self.class_vectors[c] for c in batch_classes])
          return x_inputs, batch_y  # (inputs, labels)
      else:
          return x_inputs  # solo inputs, aún como tupla

In [ ]:
def get_fine_model(pretrained_model_file, weights_path=None, learning_rate=5e-5,optimizer_name='adam'):
    # with mirrored_strategy.scope():
    model_fine = generate_bert_with_pretrained(
        pretrained_model_file, len(classes), weights_path=weights_path)
    if optimizer_name == "adam":
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == "rmsprop":
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)
    else:
        optimizer = keras.optimizers.SGD(learning_rate=learning_rate)

    model_fine.compile(optimizer=optimizer,
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    #model_fine.compile(keras.optimizers.Adam(learning_rate),
    #                   loss='categorical_crossentropy',
    #                   metrics=['accuracy'])
    max_length = model_fine.input_shape[0][1]
    return model_fine, max_length

In [ ]:
def build_model(hp):
    global global_max_length
    hp_learning_rate = hp.Float("learning_rate", 1e-6, 1e-4, sampling="log")
    optimizer_name = hp.Choice("optimizer", ["adam", "rmsprop", "sgd"])
    model, max_length = get_fine_model(pretrained_bert, weights_path=weights_path, learning_rate=hp_learning_rate, optimizer_name=optimizer_name)
    #model.summary()
    global_max_length = max_length
    return model

In [ ]:
def build_callbacks(trial):
    # Nombre del trial para identificar los logs
    trial_id = trial.trial_id
    history_filename = f"{save_name}_trial_{trial_id}_loss_history.csv"

    checkpoint_early = EarlyStopping('val_loss', patience=2, restore_best_weights=True)
    checkpoint_log = LossHistoryLogger(filename=history_filename)

    return [checkpoint_early, checkpoint_log]

Descomprimir el modelo keras para la posterior carga de configs y weights

In [ ]:
!unzip bert_nc_trained.keras model.weights.h5
!unzip bert_nc_trained.keras config.json

Archive:  bert_nc_trained.keras
 extracting: model.weights.h5        
Archive:  bert_nc_trained.keras
 extracting: config.json             


In [ ]:
pretrained_bert= '/content/config.json'
fragments_dir = '/content/27-3_input_bert_cn'
weights_path='/content/model.weights.h5'
seq_len = 151
seq_len_like_path = None  # 'path/to/seq_len_like.pkl'
k = 3
stride = 3
batch_size = 64
epochs = 1
nr_seqs = 878208
#learning_rate = 5e-5
save_name = 'fine-tune-test-1'  # 'custom_model_name'
store_predictions = True
store_train_data = False
roc_auc = False
multi_tax = False
tax_ranks = ["Cacao", "NotCacao"]
use_defined_train_test_set = False
global_max_length = None
# ==============================

norm_weights = True

if seq_len_like_path is not None:
    seq_len_dict = pickle.load(open(seq_len_like_path, 'rb'))
    min_nr_seqs = min(map(len, seq_len_dict.values()))
    seq_len_like = []
    for k_key in seq_len_dict:
        seq_len_like.extend(np.random.choice(seq_len_dict[k_key], min_nr_seqs) // k)
else:
    seq_len_like = None

if not use_defined_train_test_set:
    x, y, y_species = load_fragments(fragments_dir, nr_seqs=nr_seqs)
    f_train_x, f_test_x, f_train_y, f_test_y = train_test_split(x, y, test_size=0.2, stratify=y)
    f_train_x, f_val_x, f_train_y, f_val_y = train_test_split(f_train_x, f_train_y, test_size=0.05, stratify=f_train_y)
else:
    f_test_x, f_test_y, f_test_y_species = load_dataset(os.path.join(fragments_dir, "test.tsv"))
    f_train_x, f_train_y, f_train_y_species = load_dataset(os.path.join(fragments_dir, "train.tsv"))
    classes = ["NotCacao", "Cacao"]


#model, max_length = get_fine_model(pretrained_bert, weights_path=weights_path)
#if seq_len > max_length:
#    warning(f'desired seq len ({seq_len}) is higher than possible ({max_length}), setting to {max_length}')
#    seq_len = max_length

generator_args = {
    'max_seq_len': global_max_length, 'k': k, 'stride': stride,
    'batch_size': batch_size, 'window': True,
    'seq_len_like': seq_len_like
}

tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=20,
    directory="kt_tuning",
    project_name="bert_finetune"
)

train_generator = FragmentGenerator(f_train_x, f_train_y, seq_len, **generator_args)
val_generator = FragmentGenerator(f_val_x, f_val_y, seq_len, **generator_args)

checkpoint1 = EarlyStopping('val_loss', patience=2, restore_best_weights=True)
#checkpoint2 = LossHistoryLogger(filename=save_name + str(time.time()) + '_loss_history.csv')

callbacks_list = [checkpoint1]

tuner.search(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    callbacks=callbacks_list
)

name = save_name if save_name else ""

Trial 20 Complete [00h 10m 12s]
val_accuracy: 0.9037415981292725

Best val_accuracy So Far: 0.9459545612335205
Total elapsed time: 03h 12m 02s


In [ ]:
best_hp = tuner.get_best_hyperparameters(1)[0]
best_lr = best_hp.get("learning_rate")
best_optimizer = best_hp.get("optimizer")

# Guardamos los mejores hiperparámetros encontrados
with open("mejores_hiperparametros.txt", "w") as f:
    f.write("Mejor learning rate encontrado: {:.6f}\\n".format(best_lr))
    f.write("Mejor optimizer encontrado: "+best_optimizer)
    f.write(" Este valor fue seleccionado por lograr la mejor exactitud en validación durante el fine-tuning.\\n")


Código para descargar los resultados del tuner

In [ ]:
import shutil
from google.colab import files
folder_path = 'kt_tuning'
# Comprime la carpeta en un archivo .zip
shutil.make_archive(folder_path, 'zip', folder_path)

# Descarga el archivo zip resultante
files.download(f'{folder_path}.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>